# Random LTL formula experiments

Reads the formulas written by `Random_formula_generation.ipynb` and runs each
one through MTSA's `LTLStatsService` (`tools/ltl-stats`) on every branch in
`MTSA_BRANCHES`, collecting automaton sizes and timings. Each branch gets its
own set of output files (`stats_<branch>.csv`, `results_<branch>.json`,
`timeouts_<branch>.txt`).

Branches run one after the other. Within a branch, formulas run strictly one
at a time on a single JVM: send one, wait for it to finish (or hit
`TIMEOUT_S`), then go to the next.

Any branch that is not the one currently checked out in `REPO_ROOT` is run
from its own `git worktree` under `WORKTREE_ROOT`, so the working tree in
`REPO_ROOT` is never touched.

In [1]:
import csv
import json
import os
import signal
import subprocess
from statistics import mean
from time import time

from ltl_stats_client import LTLStats, LTLStatsError

In [2]:
# MTSA branches to test, in order; each must ship tools/ltl-stats
# (e.g. masterTests, pLTLTests)
MTSA_BRANCHES = ["masterTests", "pLTLpdTests"]

REPO_ROOT = "/home/Lakso/Programas/mtsa"
PROJECT_SUBDIR = os.path.join("maven-root", "mtsa")

# where the worktrees for branches other than the checked-out one go
WORKTREE_ROOT = "/tmp/mtsa-ltlsat"

# Per-formula wall-clock limit. A formula that takes longer is abandoned (its
# JVM is killed and a fresh one started), recorded as timed out, and the run
# moves on to the next formula.
TIMEOUT_S = 15 * 60

# Input (from Random_formula_generation.ipynb)
FORMULAS_FILE = "formulas.txt"

FILE_SUFFIX = ""


# Output files, one set per branch
def output_files(branch):
    """Returns (stats_file, results_file, timeouts_file) for `branch`."""
    tag = branch + FILE_SUFFIX
    return f"stats_{tag}.csv", f"results_{tag}.json", f"timeouts_{tag}.txt"

In [3]:
# Locate (or create) the checkout of a branch and build it once up front,
# so the build time is not charged to the first formula.
def prepare_branch(branch):
    """Returns (mtsa_dir, short_commit) for `branch`, built and ready to run."""
    current_branch = subprocess.run(
        ["git", "-C", REPO_ROOT, "rev-parse", "--abbrev-ref", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()

    if branch == current_branch:
        mtsa_dir = os.path.join(REPO_ROOT, PROJECT_SUBDIR)
    else:
        worktree = os.path.join(WORKTREE_ROOT, branch)
        if not os.path.exists(os.path.join(worktree, ".git")):
            # drop stale registrations (e.g. /tmp was wiped) before re-adding
            subprocess.run(["git", "-C", REPO_ROOT, "worktree", "prune"], check=True)
            os.makedirs(WORKTREE_ROOT, exist_ok=True)
            subprocess.run(["git", "-C", REPO_ROOT, "worktree", "add", worktree, branch],
                           check=True)
        mtsa_dir = os.path.join(worktree, PROJECT_SUBDIR)

    server_script = os.path.join(mtsa_dir, "tools", "ltl-stats", "ltl-stats-server.sh")
    if not os.path.isfile(server_script):
        raise RuntimeError(f"{branch} does not ship tools/ltl-stats ({server_script})")

    # running the launcher with no input builds (if needed), starts, and exits
    build_start = time()
    subprocess.run(["bash", server_script], cwd=mtsa_dir, stdin=subprocess.DEVNULL,
                   stdout=subprocess.DEVNULL, check=True)
    head = subprocess.run(["git", "-C", mtsa_dir, "rev-parse", "--short", "HEAD"],
                          capture_output=True, text=True, check=True).stdout.strip()
    print(f"{branch} @ {head} ready in {time() - build_start:.1f}s ({mtsa_dir})")
    return mtsa_dir, head

In [4]:
formulas = []  # (N, L, P, index, formula)
with open(FORMULAS_FILE) as fh:
    for line in fh:
        if line.strip():
            n, L, p, i, formula = line.rstrip("\n").split("\t", 4)
            formulas.append((int(n), int(L), float(p), int(i), formula))

N_VALUES = sorted({f[0] for f in formulas})
L_VALUES = sorted({f[1] for f in formulas})
P_VALUES = sorted({f[2] for f in formulas})
print(f"{len(formulas)} formulas, N in {N_VALUES}, L in {L_VALUES}, P in {P_VALUES}")

10800 formulas, N in [1, 2, 3], L in [5, 10, 15, 20, 25, 30, 35, 40, 45], P in [0.3333333333333333, 0.5, 0.7, 0.95]


In [5]:
# Run every formula, one after the other, collecting automaton stats and timings.
#
# svc.stats() blocks on a pipe read; SIGALRM interrupts that read after
# TIMEOUT_S, and the JVM is then killed and replaced by a fresh one.

class FormulaTimeout(Exception):
    pass


def _on_alarm(signum, frame):
    raise FormulaTimeout()


def _blank_rec(n, L, p, i, formula, *, error, timed_out):
    return {
        "N": n, "L": L, "P": p, "index": i, "formula": formula,
        "ok": False, "error": error, "is_property": None,
        "gba_states": None, "gba_transitions": None,
        "buchi_states": None, "buchi_transitions": None,
        "timed_out": timed_out,
    }


def run_formula(svc, n, L, p, i, formula):
    """Returns (record, svc_still_usable)."""
    signal.alarm(TIMEOUT_S)
    try:
        res = svc.stats(formula, raise_on_error=False)
    except FormulaTimeout:
        print(f"  TIMEOUT N={n} L={L} P={p} #{i} after {TIMEOUT_S}s: {formula}")
        return _blank_rec(n, L, p, i, formula, error=f"timed out after {TIMEOUT_S}s",
                          timed_out=True), False
    except LTLStatsError as e:
        return _blank_rec(n, L, p, i, formula, error=f"worker crashed: {e}",
                          timed_out=False), False
    finally:
        signal.alarm(0)
    return {
        "N": n, "L": L, "P": p, "index": i, "formula": formula,
        "ok": res.ok, "error": res.error, "is_property": res.is_property,
        "gba_states": res.gba.states if res.gba else None,
        "gba_transitions": res.gba.transitions if res.gba else None,
        "buchi_states": res.buchi.states if res.buchi else None,
        "buchi_transitions": res.buchi.transitions if res.buchi else None,
        "timed_out": False,
    }, True


def stop(svc):
    if svc is None:
        return
    try:
        svc.close()
    except Exception:
        pass
    svc.kill()


def run_experiment(mtsa_dir):
    """Runs every formula on the MTSA checkout in `mtsa_dir`.

    Returns (all_results, combo_times, total_elapsed): one record per formula,
    and (N, L, P) -> seconds spent on that combination's formulas.
    """
    all_results = []
    combo_times = {}

    old_handler = signal.signal(signal.SIGALRM, _on_alarm)
    svc = None
    sweep_start = time()
    last_nl = None
    try:
        for n, L, p, i, formula in formulas:
            if (n, L) != last_nl:
                print(f"N={n} L={L} (t={time() - sweep_start:.1f}s)", flush=True)
                last_nl = (n, L)
            if svc is None:
                svc = LTLStats.spawn(mtsa_dir)
            run_start = time()
            rec, usable = run_formula(svc, n, L, p, i, formula)
            rec["run_time_s"] = time() - run_start
            if not usable:
                svc.kill()
                svc = None   # a fresh JVM is spawned for the next formula
            all_results.append(rec)
            combo_times[(n, L, p)] = combo_times.get((n, L, p), 0.0) + rec["run_time_s"]
        total_elapsed = time() - sweep_start
    finally:
        # Always tear the JVM down -- otherwise an interrupt orphans it and it
        # keeps holding gigabytes.
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)
        stop(svc)

    for (n, L, p), t in combo_times.items():
        rows = [r for r in all_results if r["N"] == n and r["L"] == L and r["P"] == p]
        n_to = sum(r["timed_out"] for r in rows)
        print(f"N={n} L={L} P={p}: {len(rows)} formulas in {t:.4f}s"
              + (f" ({n_to} timed out)" if n_to else ""))

    n_timeouts = sum(r["timed_out"] for r in all_results)
    print(f"Terminado en {total_elapsed}s ({n_timeouts} formulas timed out)")
    return all_results, combo_times, total_elapsed

In [6]:
def save_results(branch, head, all_results, combo_times):
    stats_file, results_file, timeouts_file = output_files(branch)

    # Save the formulas that hit the timeout, so they can be inspected / rerun.
    timed_out = [r for r in all_results if r.get("timed_out")]
    with open(timeouts_file, "w") as fh:
        for rec in timed_out:
            fh.write(f"{rec['N']}\t{rec['L']}\t{rec['P']}\t{rec['index']}\t{rec['formula']}\n")

    # Save the per-formula stats for all automata (gba + buchi) plus run time as CSV.
    fieldnames = [
        "N", "L", "P", "index", "formula", "ok", "error", "is_property",
        "gba_states", "gba_transitions", "buchi_states", "buchi_transitions",
        "run_time_s", "timed_out",
    ]
    with open(stats_file, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(all_results)

    # Save everything (stats + per-run times + per-combination times) as JSON.
    with open(results_file, "w") as fh:
        json.dump({
            "mtsa_branch": branch,
            "mtsa_commit": head,
            "N_values": N_VALUES,
            "L_values": L_VALUES,
            "P_values": P_VALUES,
            "timeout_s": TIMEOUT_S,
            "combination_times_s": {f"N={n},L={L},P={p}": t for (n, L, p), t in combo_times.items()},
            "total_time_s": sum(combo_times.values()),
            "timed_out": [
                {"N": r["N"], "L": r["L"], "P": r["P"], "index": r["index"], "formula": r["formula"]}
                for r in timed_out
            ],
            "results": all_results,
        }, fh, indent=2)

    print(f"Wrote {stats_file}, {results_file}, {timeouts_file}")
    print(f"Total: {len(all_results)} runs, {sum(combo_times.values()):.4f}s, "
          f"{len(timed_out)} timed out")

In [7]:
# Quick per-combination summary.
def summarize(all_results, combo_times):
    for n in N_VALUES:
        for L in L_VALUES:
            for p in P_VALUES:
                rows = [r for r in all_results
                        if r["N"] == n and r["L"] == L and r["P"] == p and r["ok"]]
                if not rows:
                    print(f"N={n} L={L} P={p}: no successful runs")
                    continue
                print(
                    f"N={n} L={L} P={p}: "
                    f"gba states avg {mean(r['gba_states'] for r in rows):.2f}, "
                    f"buchi states avg {mean(r['buchi_states'] for r in rows):.2f}, "
                    f"properties {sum(bool(r['is_property']) for r in rows)}/{len(rows)}, "
                    f"run time avg {mean(r['run_time_s'] for r in rows):.4f}s, "
                    f"combo time {combo_times[(n, L, p)]:.4f}s"
                )

In [8]:
# Run the whole experiment on each branch in turn. A branch that fails to set
# up or crashes is reported and skipped; its results are not saved.
branch_results = {}   # branch -> (head, all_results, combo_times)
failed_branches = {}  # branch -> error

for branch in MTSA_BRANCHES:
    print(f"\n===== {branch} =====", flush=True)
    try:
        mtsa_dir, head = prepare_branch(branch)
        all_results, combo_times, _ = run_experiment(mtsa_dir)
    except Exception as e:
        print(f"{branch} FAILED: {e!r}")
        failed_branches[branch] = e
        continue
    save_results(branch, head, all_results, combo_times)
    summarize(all_results, combo_times)
    branch_results[branch] = (head, all_results, combo_times)

print(f"\nDone: {len(branch_results)}/{len(MTSA_BRANCHES)} branches ran"
      + (f", failed: {', '.join(failed_branches)}" if failed_branches else ""))


===== masterTests =====
masterTests @ 0331a27f0 ready in 0.5s (/tmp/mtsa-ltlsat/masterTests/maven-root/mtsa)
N=1 L=5 (t=0.0s)
N=1 L=10 (t=1.5s)
N=1 L=15 (t=2.6s)
N=1 L=20 (t=3.8s)
N=1 L=25 (t=5.0s)
N=1 L=30 (t=6.3s)
N=1 L=35 (t=8.2s)
N=1 L=40 (t=12.8s)
N=1 L=45 (t=20.3s)
N=2 L=5 (t=155.1s)
N=2 L=10 (t=155.5s)
N=2 L=15 (t=156.1s)
N=2 L=20 (t=156.7s)
N=2 L=25 (t=158.1s)
N=2 L=30 (t=161.7s)
N=2 L=35 (t=177.5s)
N=2 L=40 (t=334.3s)
N=2 L=45 (t=534.4s)
  TIMEOUT N=2 L=45 P=0.95 #11 after 900s: (((X a) W ((a U b) W (a W ((b W b) U ((X b) U (b U b)))))) U (((((X a) U a) U ((X b) W (X a))) W ((a W a) W (((X b) U a) W a))) W (a W a)))
N=3 L=5 (t=2029.3s)
N=3 L=10 (t=2030.3s)
N=3 L=15 (t=2031.4s)
N=3 L=20 (t=2032.8s)
N=3 L=25 (t=2037.9s)
N=3 L=30 (t=2061.5s)
N=3 L=35 (t=2209.4s)
N=3 L=40 (t=2894.3s)
  TIMEOUT N=3 L=40 P=0.5 #62 after 900s: ((X (((c W a) U a) W ((a U a) U (X b)))) W (((((X (X c)) U a) U ((X c) U a)) U (X (!(X c)))) W ((X b) W (b W ((c W b) U b)))))
  TIMEOUT N=3 L=40 P=0.95 #32 a

building project + classpath (a925637268efafe2bd75be49f9b39a325bc40161)...


pLTLpdTests @ a92563726 ready in 208.4s (/home/Lakso/Programas/mtsa/maven-root/mtsa)
N=1 L=5 (t=0.0s)
N=1 L=10 (t=1.3s)
N=1 L=15 (t=2.7s)
N=1 L=20 (t=4.8s)
N=1 L=25 (t=73.2s)


Exception in thread "main" java.lang.OutOfMemoryError: Java heap space


N=1 L=30 (t=526.6s)
N=1 L=35 (t=976.4s)


...

  TIMEOUT N=3 L=45 P=0.7 #21 after 900s: (!((((((a && (c U b)) U (X c)) U (X (!a))) && ((!a) W b)) W c) U (a && (((c && (!a)) U (a U b)) U (((!a) W c) && (a U (((!c) W a) U a)))))))
  TIMEOUT N=3 L=45 P=0.7 #57 after 900s: (((((c W a) W (((c || (X c)) U a) W c)) U a) || ((b U (a W (!c))) U (!((a || a) W b)))) || ((a U c) U ((c U a) W ((X ((!b) W a)) W (X c)))))


Exception in thread "main" java.lang.OutOfMemoryError: Java heap space


  TIMEOUT N=3 L=45 P=0.7 #72 after 900s: ((a W ((b W (!c)) U b)) U ((X (b W (X (((!a) U b) || (c W b))))) || ((a U (b && b)) || (((b || (c U b)) W (c U (a && b))) U (a W (X (!c)))))))


Exception in thread "main" java.lang.OutOfMemoryError: Java heap space
	at ltsa.lts.ltl.LTLStatsService.quote(LTLStatsService.java:237)
	at ltsa.lts.ltl.LTLStatsService.key(LTLStatsService.java:229)
	at ltsa.lts.ltl.LTLStatsService.statsAsJson(LTLStatsService.java:154)
	at ltsa.lts.ltl.LTLStatsService.pipe(LTLStatsService.java:108)
	at ltsa.lts.ltl.LTLStatsService.main(LTLStatsService.java:79)
Exception in thread "main" java.lang.OutOfMemoryError: Java heap space
Exception in thread "main" java.lang.OutOfMemoryError: Java heap space
Exception in thread "main" java.lang.OutOfMemoryError: Java heap space
Exception in thread "main" java.lang.OutOfMemoryError: Java heap space
Exception in thread "main" java.lang.OutOfMemoryError: Java heap space
Exception in thread "main" java.lang.OutOfMemoryError: Java heap space


N=1 L=5 P=0.3333333333333333: 100 formulas in 0.3729s
N=1 L=5 P=0.5: 100 formulas in 0.2884s
N=1 L=5 P=0.7: 100 formulas in 0.3036s
N=1 L=5 P=0.95: 100 formulas in 0.3117s
N=1 L=10 P=0.3333333333333333: 100 formulas in 0.3206s
N=1 L=10 P=0.5: 100 formulas in 0.3401s
N=1 L=10 P=0.7: 100 formulas in 0.3843s
N=1 L=10 P=0.95: 100 formulas in 0.3787s
N=1 L=15 P=0.3333333333333333: 100 formulas in 0.3551s
N=1 L=15 P=0.5: 100 formulas in 0.3974s
N=1 L=15 P=0.7: 100 formulas in 0.3599s
N=1 L=15 P=0.95: 100 formulas in 1.0003s
N=1 L=20 P=0.3333333333333333: 100 formulas in 4.4452s
N=1 L=20 P=0.5: 100 formulas in 0.8733s
N=1 L=20 P=0.7: 100 formulas in 14.6943s
N=1 L=20 P=0.95: 100 formulas in 48.3416s
N=1 L=25 P=0.3333333333333333: 100 formulas in 0.8590s
N=1 L=25 P=0.5: 100 formulas in 33.4654s
N=1 L=25 P=0.7: 100 formulas in 178.9530s
N=1 L=25 P=0.95: 100 formulas in 240.1626s
N=1 L=30 P=0.3333333333333333: 100 formulas in 4.8407s
N=1 L=30 P=0.5: 100 formulas in 39.6686s
N=1 L=30 P=0.7: 100 f